### STEP 2

In [3]:
import zipfile

with zipfile.ZipFile("archive (1).zip", 'r') as zip_ref:
    zip_ref.extractall()

In [13]:
import pandas as pd
import re
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


### STEP 3

In [5]:
print("Shape:", df.shape)
print("\nColumns:", df.columns)
print("\nClass Distribution:")
print(df['sentiment'].value_counts())

print("\nSample Data:")
print(df.head())

Shape: (50000, 2)

Columns: Index(['review', 'sentiment'], dtype='object')

Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample Data:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


### STEP 4

In [6]:
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

In [18]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\WIN11\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\WIN11\AppData\Roaming\nltk_data...


In [28]:
lemmatizer = WordNetLemmatizer()

### STEP 5

In [7]:
from sklearn.model_selection import train_test_split

X = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [33]:
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\WIN11\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


### STEP 6

In [34]:
import re
from nltk.corpus import stopwords

def preprocess_text(text):
    if not isinstance(text, str) or text.strip() == "":
        return [], ""
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # Remove emails
    text = re.sub(r"\S+@\S+", "", text)
    
    # Remove numbers
    text = re.sub(r"\d+", "", text)
    
    # Lowercase
    text = text.lower()
    
    # Handle repeated characters
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    
    # Remove special characters & emojis
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenization
    tokens = text.split()
    
    tokens = [word for word in tokens if word not in stop_words or word in ['no', 'not']]
    
    # Remove short tokens
    tokens = [word for word in tokens if len(word) > 2 or word in ['no', 'not']]
    
    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Reconstruct sentence
    cleaned_sentence = " ".join(tokens)
    
    return tokens, cleaned_sentence

In [14]:
X_train_clean = X_train.apply(lambda x: preprocess_text(x)[1])
X_test_clean = X_test.apply(lambda x: preprocess_text(x)[1])

In [15]:
print(X_train_clean.iloc[0])

thats what kept asking myself during the many fights screaming matches swearing and general mayhem that permeate the minutes the comparisons also stand when you think the onedimensional characters who have little depth that virtually impossible care what happens them they are just badly written cyphers for the director hang his multicultural beliefs topic that has been done much better other dramas both and the cinemabr must confess not really one for spotting bad performances during film but must said that nichola burley the heroines slutty best friend and wasim zakir the nasty bullying brother were absolutely terrible dont know what acting school they graduated from but was them apply for full refund post haste only samina awan the lead role manages impress cast socalled british talent that well probably never hear from again least thats the hope next time hire different scoutbr another intriguing thought the hideously fashionable soundtrack featuring the likes snow patrol ian brown 

### STEP 7

In [16]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train_clean)
X_test_bow = bow.transform(X_test_clean)

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train_clean)
X_test_tfidf = tfidf.transform(X_test_clean)

## STEP 8

### Logistic Regression ###

In [35]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()

lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

### Naive Bayes ###

In [36]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()

nb.fit(X_train_tfidf, y_train)

y_pred_nb = nb.predict(X_test_tfidf)

### Decision Tree ###

In [37]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()

dt.fit(X_train_tfidf, y_train)

y_pred_dt = dt.predict(X_test_tfidf)

### STEP 9 

In [41]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [43]:
def evaluate_model(y_test, y_pred):
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("-" * 40)

In [44]:
print("Logistic Regression:")
evaluate_model(y_test, y_pred_lr)

print("Naive Bayes:")
evaluate_model(y_test, y_pred_nb)

print("Decision Tree:")
evaluate_model(y_test, y_pred_dt)

Logistic Regression:
Accuracy: 0.8911
Precision: 0.8821594427244582
Recall: 0.9047430045643977
F1 Score: 0.8933085137650632
----------------------------------------
Naive Bayes:
Accuracy: 0.8531
Precision: 0.856003191065018
Recall: 0.851756300853344
F1 Score: 0.8538744653337312
----------------------------------------
Decision Tree:
Accuracy: 0.7119
Precision: 0.7154552715654952
Recall: 0.7110537805120064
F1 Score: 0.7132477356424803
----------------------------------------


In [45]:
import pandas as pd

results = {
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_nb),
        precision_score(y_test, y_pred_dt)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_nb),
        recall_score(y_test, y_pred_dt)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_nb),
        f1_score(y_test, y_pred_dt)
    ]
}

df_results = pd.DataFrame(results)
df_results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.8911,0.882159,0.904743,0.893309
1,Naive Bayes,0.8531,0.856003,0.851756,0.853874
2,Decision Tree,0.7119,0.715455,0.711054,0.713248


## Comparison & Insights

### Preprocessing Insights:
- Lowercasing ensured uniform text representation
- Removal of stopwords reduced noise while retaining important words like "not"
- Lemmatization improved word normalization and reduced dimensionality
- Removing special characters and URLs cleaned irrelevant data

### Feature Engineering:
- Bag of Words captures word frequency but ignores importance
- TF-IDF performed better as it gives importance to meaningful words and reduces common word impact

### Model Comparison:
- Logistic Regression performed the best (~89% accuracy)
- Naive Bayes was efficient and performed well (~85%)
- Decision Tree performed poorly (~71%) due to overfitting on sparse text data

### Best Combination:
TF-IDF + Logistic Regression gave the best results

### Trade-offs:
- Logistic Regression: High accuracy but slightly slower
- Naive Bayes: Fast but less accurate
- Decision Tree: Interpretable but not suitable for high-dimensional text

### Final Conclusion:
Linear models are better suited for NLP tasks, especially when using TF-IDF features.

## Summary

In this project, an end-to-end sentiment analysis system was built using NLP techniques and machine learning models.

The preprocessing pipeline cleaned and normalized the text effectively. TF-IDF feature extraction provided better performance compared to Bag of Words. Among the models, Logistic Regression achieved the best performance, making it the most suitable choice for sentiment classification.

This project demonstrates how proper preprocessing and feature engineering significantly impact model performance in NLP tasks.